# TFS-IMM MIMIC-IV Preprocessing (Final)

Design principles:

- No normalization in preprocessing.
- No train/validation/test split in preprocessing.
- Preserve irregular timestamps and missingness.
- Main forecasting task: 24h context -> 24h future prediction.
- Prefer larger patient cohort over long single-patient trajectories.
- Text is filtered by availability time to avoid future leakage.
- Large MIMIC tables are processed with chunk loading.

Downstream files should handle:
- patient-level 6:2:2 split
- normalization
- dataloader construction
- training


## Performance optimization notes

This version is optimized for large-scale TFS-IMM cohort construction.

Changes:
- Removed row-wise pandas operations (`iterrows`, `.at`).
- Replaced label expansion with vectorized `pivot_table`.
- Removed repeated `pd.concat` inside chunk loops.
- Kept preprocessing only; no normalization or train/val/test split.


# TFS-IMM MIMIC-IV Preprocessing

This notebook is adapted from the TIME-IMM MIMIC-IV preprocessing pipeline for the **TFS-IMM Medical Core** task.

## Core protocol

- **Multivariate irregular time-series forecasting**
- **24 h context → 24 h forecast**
- **One eligible ICU episode per patient** in the Core benchmark
- Prefer **more independent patients** over many overlapping windows from a few long stays
- Use a **fixed feature space** while preserving native irregular timestamps and missingness
- Text is restricted to the **history window only** (`0 <= note_time < 24 h`) to avoid future leakage
- Build two cohorts:
  - `processed_full`: numeric-valid samples, text may be absent
  - `processed_mm`: same construction, but requires at least one history note
- Split at the **patient level** before model training
- No overlapping sliding windows are generated in this notebook

The original TIME-IMM preprocessing logic for event cleaning is retained where possible.


Please make sure to download these two projects:
- [MIMIC-IV-Note](https://physionet.org/content/mimic-iv-note/2.2/)
- [MIMIC-IV](https://physionet.org/content/mimiciv/3.1/)

So under the folder MIMIC, you would have:
```
data/mimiciv/3.1/
data/mimic-iv-note/2.2/
```

All Time Series Processing

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta
import numpy as np
from tqdm import tqdm
import shutil

In [ ]:
from pathlib import Path

# =============================================================================
# TFS-IMM CONFIGURATION
# =============================================================================
PROJECT_ROOT = Path.cwd()
MIMIC_IV_ROOT = PROJECT_ROOT / "data" / "mimiciv" / "3.1"
MIMIC_NOTE_ROOT = PROJECT_ROOT / "data" / "mimic-iv-note" / "2.2" / "note"

# Main forecasting task
CONTEXT_HOURS = 24
PRED_HOURS = 24
TOTAL_WINDOW_HOURS = CONTEXT_HOURS + PRED_HOURS

# Cohort definition
MIN_AGE = 18
MIN_ICU_STAY_HOURS = TOTAL_WINDOW_HOURS

# Optional upper bound. Keep None for the main large-cohort benchmark.
# Set e.g. 30 if you want a t-PatchGNN-like sensitivity analysis.
MAX_ICU_STAY_DAYS = None

# Sample quality filters.
# These are deliberately configurable; inspect the generated statistics before
# tightening them for the final paper.
MIN_HISTORY_OBSERVATIONS = 20
MIN_TARGET_OBSERVATIONS = 20
MIN_HISTORY_VARIABLES = 5
MIN_TARGET_VARIABLES = 5

# Multimodal subset
MIN_TEXT_NOTES_MM = 1
TEXT_SOURCE = "radiology"

# Remove stale output cohort folders before rebuilding.
RESET_OUTPUT_DIRS = True


if not MIMIC_IV_ROOT.exists():
    raise FileNotFoundError(f"MIMIC-IV directory not found: {MIMIC_IV_ROOT.resolve()}")
if not MIMIC_NOTE_ROOT.exists():
    raise FileNotFoundError(f"MIMIC-IV-Note directory not found: {MIMIC_NOTE_ROOT.resolve()}")

print(f"MIMIC-IV: {MIMIC_IV_ROOT.resolve()}")
print(f"MIMIC-IV-Note: {MIMIC_NOTE_ROOT.resolve()}")
print(f"Task: {CONTEXT_HOURS}h context -> {PRED_HOURS}h forecast")


In [ ]:
adm = pd.read_csv(MIMIC_IV_ROOT / "hosp" / "admissions.csv.gz")
patients_df = pd.read_csv(MIMIC_IV_ROOT / "hosp" / "patients.csv.gz")
icu_stays = pd.read_csv(MIMIC_IV_ROOT / "icu" / "icustays.csv.gz")

for col in ["admittime", "dischtime"]:
    adm[col] = pd.to_datetime(adm[col])

for col in ["intime", "outtime"]:
    icu_stays[col] = pd.to_datetime(icu_stays[col])

print(f"Admissions: {len(adm):,}")
print(f"Patients: {patients_df['subject_id'].nunique():,}")
print(f"ICU stays: {len(icu_stays):,}")


In [ ]:
# =============================================================================
# PATIENT-LEVEL COHORT
# Instead of requiring exactly one hospital admission and >50-day LOS,
# keep the first ICU stay per patient that is long enough for 24h -> 24h.
# =============================================================================
cohort = (
    icu_stays[
        ["subject_id", "hadm_id", "stay_id", "intime", "outtime"]
    ]
    .merge(
        adm[["subject_id", "hadm_id", "admittime", "dischtime"]],
        on=["subject_id", "hadm_id"],
        how="inner",
    )
    .merge(
        patients_df[["subject_id", "anchor_age"]],
        on="subject_id",
        how="inner",
    )
)

cohort["icu_los_hours"] = (
    cohort["outtime"] - cohort["intime"]
).dt.total_seconds() / 3600.0

eligible = cohort[
    (cohort["anchor_age"] >= MIN_AGE)
    & (cohort["icu_los_hours"] >= MIN_ICU_STAY_HOURS)
].copy()

if MAX_ICU_STAY_DAYS is not None:
    eligible = eligible[
        eligible["icu_los_hours"] <= MAX_ICU_STAY_DAYS * 24
    ].copy()

# Core benchmark: one eligible ICU episode per patient.
# Choose the earliest eligible ICU stay deterministically.
eligible = eligible.sort_values(["subject_id", "intime", "stay_id"])
adm_3 = eligible.drop_duplicates("subject_id", keep="first").copy()

assert adm_3["subject_id"].is_unique
assert adm_3["hadm_id"].is_unique

print(f"Eligible patients for {TOTAL_WINDOW_HOURS}h task: {len(adm_3):,}")
print(adm_3["icu_los_hours"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]))


In [ ]:
# ICU admission time is the canonical t=0 for every sample.
# This is safer than using the first observed measurement, whose timing differs
# across patients and could shift the 24h/24h task.
ref_time = adm_3.set_index("hadm_id")["intime"].copy()

cohort_summary = adm_3[
    ["subject_id", "hadm_id", "stay_id", "intime", "outtime", "anchor_age", "icu_los_hours"]
].copy()
cohort_summary.head()


In [ ]:
Path("raw").mkdir(exist_ok=True)
cohort_summary.to_csv("raw/admissions_processed.csv", index=False)
print(f"Saved initial eligible cohort: {len(cohort_summary):,} patients")


In [ ]:
# Confirm that selected admissions have ICU chart events.
# Optimized for large patient cohorts:
# - read only hadm_id column
# - avoid repeated list/set conversions
# - avoid storing unnecessary chart rows

fn = MIMIC_IV_ROOT / "icu" / "chartevents.csv.gz"

target_hadm_ids = set(
    adm_3["hadm_id"]
    .dropna()
    .astype("int64")
    .tolist()
)

chart_hadm_ids = set()

for chunk in tqdm(
    pd.read_csv(
        fn,
        usecols=["hadm_id"],
        chunksize=2_000_000,
    ),
    desc="Checking chartevents coverage",
):
    # Remove missing admission ids first
    hadm = chunk["hadm_id"].dropna().astype("int64")

    # Keep only selected admissions
    matched = hadm[hadm.isin(target_hadm_ids)]

    if len(matched) > 0:
        chart_hadm_ids.update(matched.unique())

print(
    f"Selected admissions with chartevents: "
    f"{len(chart_hadm_ids):,}/{len(target_hadm_ids):,}"
)


In [ ]:
# Keep only patients with chart events, while preserving one episode per patient.
adm_3 = adm_3[adm_3["hadm_id"].astype(int).isin(chart_hadm_ids)].copy()
ref_time = adm_3.set_index("hadm_id")["intime"].copy()

assert adm_3["subject_id"].is_unique
assert adm_3["hadm_id"].is_unique

print(f"Patients after chartevents availability check: {len(adm_3):,}")


In [ ]:
adm_3.to_csv("raw/admissions_processed.csv", index=False)


In [ ]:
# Downstream cells use `adm` / `adm_ids`; point them to the final selected cohort.
adm = adm_3.copy()
adm_ids = list(adm_3["hadm_id"])


In [ ]:
# only choose previously selected admission ids (takes about 30 seconds)
inputs = pd.read_csv(MIMIC_IV_ROOT / "icu" / "inputevents.csv.gz")
adm_ids = list(adm_3["hadm_id"])
inputs = inputs.loc[inputs["hadm_id"].isin(adm_ids)]
inputs.head()


In [ ]:
# only keep columns of interest
inputs_small = inputs[
    [
        "subject_id",
        "hadm_id",
        "starttime",
        "endtime",
        "itemid",
        "amount",
        "amountuom",
        "rate",
        "rateuom",
        "patientweight",
        "ordercategorydescription",
    ]
]
print("Number of patients remaining in the database: ")
print(inputs_small["subject_id"].nunique())

In [ ]:
# get item ids for inputs
item_id = pd.read_csv("data/mimiciv/3.1/icu/d_items.csv.gz")
item_id_1 = item_id[["itemid", "label"]]
item_id_1.head()

inputs_small_2 = pd.merge(inputs_small, item_id_1, on="itemid")
inputs_small_2.head()
print("Number of patients remaining in the database: ")
print(inputs_small_2["subject_id"].nunique())

In [ ]:
# For each item, evaluate the number of patients who have been given this item
# Select only the inputs with highest occurence
pat_for_item = inputs_small_2.groupby("label")["subject_id"].nunique()
frequent_labels = pat_for_item.sort_values(ascending=False)[:50]
inputs_small_3 = inputs_small_2.loc[
    inputs_small_2["label"].isin(list(frequent_labels.index))
].copy()

print("Number of patients remaining in the database: ")
print(inputs_small_3["subject_id"].nunique())

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print(inputs_small_3.groupby("label")["amountuom"].value_counts())

In [ ]:
##### Cleaning the Cefazolin (remove the ones that are not in dose unit)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["itemid"] == 225850) & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Cefepime (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Cefepime")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Ceftriaxone (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Ceftriaxone")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Ciprofloxacin (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Ciprofloxacin")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Famotidine (Pepcid) (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Famotidine (Pepcid)")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Fentanyl (Concentrate) (remove the non mg)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Fentanyl (Concentrate)")
        & (inputs_small_3["amountuom"] != "mg")
    ].index
).copy()
inputs_small_3.loc[
    (inputs_small_3["label"] == "Fentanyl (Concentrate)")
    & (inputs_small_3["amountuom"] == "mg"),
    "amount",
] *= 1000
inputs_small_3.loc[
    (inputs_small_3["label"] == "Fentanyl (Concentrate)")
    & (inputs_small_3["amountuom"] == "mg"),
    "amountuom",
] = "mcg"
# Cleaning the Heparin Sodium (Prophylaxis) (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Heparin Sodium (Prophylaxis)")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Hydromorphone (Dilaudid) (remove the non mg)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Hydromorphone (Dilaudid)")
        & (inputs_small_3["amountuom"] != "mg")
    ].index
).copy()
# Cleaning the Magnesium Sulfate (remove the non grams)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Magnesium Sulfate")
        & (inputs_small_3["amountuom"] != "grams")
    ].index
).copy()
# Cleaning the Propofol (remove the non mg)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Propofol") & (inputs_small_3["amountuom"] != "mg")
    ].index
).copy()
# Cleaning the Metoprolol (remove the non mg)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Metoprolol")
        & (inputs_small_3["amountuom"] != "mg")
    ].index
).copy()
# Cleaning the Piperacillin/Tazobactam (Zosyn) (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Piperacillin/Tazobactam (Zosyn)")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Metronidazole (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Metronidazole")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Ranitidine (Prophylaxis)(remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Ranitidine (Prophylaxis)")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Vancomycin (remove the non dose)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Vancomycin")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()
# Cleaning the Fentanyl. Put the mg to mcg
inputs_small_3.loc[
    (inputs_small_3["itemid"] == 221744) & (inputs_small_3["amountuom"] == "mg"),
    "amount",
] *= 1000
inputs_small_3.loc[
    (inputs_small_3["itemid"] == 221744) & (inputs_small_3["amountuom"] == "mg"),
    "amountuom",
] = "mcg"
# Cleaning of the Pantoprazole (Protonix)
# divide in two (drug shot or continuous treatment and create a new item id for the continuous version)
inputs_small_3.loc[
    (inputs_small_3["itemid"] == 225910)
    & (inputs_small_3["ordercategorydescription"] == "Continuous Med"),
    "label",
] = "Pantoprazole (Protonix) Continuous"
inputs_small_3.loc[
    (inputs_small_3["itemid"] == 225910)
    & (inputs_small_3["ordercategorydescription"] == "Continuous Med"),
    "itemid",
] = 2217441
# remove the non dose from the drug shot version
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Pantoprazole (Protonix)")
        & (inputs_small_3["amountuom"] != "dose")
    ].index
).copy()

In [ ]:
# Additional Preprocessing for MIMIC 4 items
# Cleaning the Acetaminophen-IV (keep mg)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Acetaminophen-IV")
        & (inputs_small_3["amountuom"] != "mg")
    ].index
).copy()

# Cleaning the D5 1/2NS (keep ml)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "D5 1/2NS") & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

# Cleaning the Dexmedetomidine (Precedex) (cast all to mg)
inputs_small_3.loc[
    (inputs_small_3["label"] == "Dexmedetomidine (Precedex)")
    & (inputs_small_3["amountuom"] == "mcg"),
    "amount",
] /= 1000
inputs_small_3.loc[
    (inputs_small_3["label"] == "Dexmedetomidine (Precedex)")
    & (inputs_small_3["amountuom"] == "mcg"),
    "amountuom",
] = "mg"

# Cleaning the LR
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "LR") & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

# Cleaning the NaCl 0.9%
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "NaCl 0.9%") & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

# Cleaning the OR Crystalloid Intake
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "OR Crystalloid Intake")
        & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

# Cleaning the PO Intake
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "PO Intake") & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

# Cleaning the Pre-Admission/Non-ICU Intake
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Pre-Admission/Non-ICU Intake")
        & (inputs_small_3["amountuom"] != "ml")
    ].index
).copy()

In [ ]:
# Verify
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    print(inputs_small_3.groupby("label")["amountuom"].value_counts())

In [ ]:
# same thing for inputs given in rates
inputs_small_3.groupby("label")["rateuom"].value_counts()

In [ ]:
# Cleaning of Dextrose 5%  (remove the non mL/hour)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Dextrose 5%")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()
# Cleaning of Magnesium Sulfate (Bolus)  (remove the non mL/hour)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Magnesium Sulfate (Bolus)")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()
# Cleaning of NaCl 0.9% (remove the non mL/hour)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "NaCl 0.9%")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()
# Cleaning of Piggyback (remove the non mL/hour)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Piggyback")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()
# Cleaning of Packed Red Bllod Cells (remove the non mL/hour)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Packed Red Blood Cells")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()

# additional cleaning for mimic4
# Cleaning of Acetaminophen-IV
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Acetaminophen-IV")
        & (inputs_small_3["rateuom"] != "mg/min")
    ].index
).copy()

# Cleaning of Fentanyl (Concentrate)
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Fentanyl (Concentrate)")
        & (inputs_small_3["rateuom"] != "mcg/hour")
    ].index
).copy()

# Cleaning of Phenylephrine
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Phenylephrine")
        & (inputs_small_3["rateuom"] != "mcg/kg/min")
    ].index
).copy()

# Cleaning of Sterile Water
inputs_small_3 = inputs_small_3.drop(
    inputs_small_3.loc[
        (inputs_small_3["label"] == "Sterile Water")
        & (inputs_small_3["rateuom"] != "mL/hour")
    ].index
).copy()


# Check if a single unit per drug
inputs_small_3.groupby("label")["rateuom"].value_counts()

We now split the entries which are spread in time.
We chose the duration window for the sampling. here we choose 30 minutes. So every entry which has a rate and with duration larger than 1 hour, we split it into fixed times injections.

In [ ]:
df_temp = inputs_small_3.loc[
    (inputs_small_3["rate"].notnull())
    & (inputs_small_3["rateuom"].str.contains("mg/min"))
].copy()
df_temp["computed_amount"] = df_temp["rate"] * (
    (
        pd.to_datetime(df_temp["endtime"]) - pd.to_datetime(df_temp["starttime"])
    ).dt.total_seconds()
    / 60
)

# Check with a 0.01 tolerance
assert (
    len(df_temp.loc[(abs(df_temp["computed_amount"] - df_temp["amount"]) > 0.01)].index)
    == 0
)  # OK

# Third check the kg/min units
df_temp = inputs_small_3.loc[
    (inputs_small_3["rate"].notnull())
    & (inputs_small_3["rateuom"].str.contains("mcg/kg/min"))
].copy()
df_temp["computed_amount"] = (
    df_temp["rate"]
    * (
        (
            pd.to_datetime(df_temp["endtime"]) - pd.to_datetime(df_temp["starttime"])
        ).dt.total_seconds()
        / 60
    )
    * df_temp["patientweight"]
)

# Check with a 0.01 tolerance
assert (
    len(
        df_temp.loc[
            (abs(df_temp["computed_amount"] / 1000 - df_temp["amount"]) > 0.01)
        ].index
    )
    == 0
)  # OK

In [ ]:
inputs_small_3.head()

In [ ]:
duration_split_hours = 0.5
to_sec_fact = 3600 * duration_split_hours

# split data set in four.

# The first dataframe contains the entries with no rate but with extended duration inputs (over 0.5 hour)
df_temp1 = (
    inputs_small_3.loc[
        (
            (
                pd.to_datetime(inputs_small_3["endtime"])
                - pd.to_datetime(inputs_small_3["starttime"])
            )
            > timedelta(hours=duration_split_hours)
        )
        & (inputs_small_3["rate"].isnull())
    ]
    .copy()
    .reset_index(drop=True)
)
# The second dataframe contains the entries with no rate and low duration entries (<0.5hour)
df_temp2 = (
    inputs_small_3.loc[
        (
            (
                pd.to_datetime(inputs_small_3["endtime"])
                - pd.to_datetime(inputs_small_3["starttime"])
            )
            <= timedelta(hours=duration_split_hours)
        )
        & (inputs_small_3["rate"].isnull())
    ]
    .copy()
    .reset_index(drop=True)
)
# The third dataframe contains the entries with a rate and extended duration inputs (over 0.5 hour)
df_temp3 = (
    inputs_small_3.loc[
        (
            (
                pd.to_datetime(inputs_small_3["endtime"])
                - pd.to_datetime(inputs_small_3["starttime"])
            )
            > timedelta(hours=duration_split_hours)
        )
        & (inputs_small_3["rate"].notnull())
    ]
    .copy()
    .reset_index(drop=True)
)
# The forth dataframe contains the entries with a rate and low duration entries (< 0.5 hour)
df_temp4 = (
    inputs_small_3.loc[
        (
            (
                pd.to_datetime(inputs_small_3["endtime"])
                - pd.to_datetime(inputs_small_3["starttime"])
            )
            <= timedelta(hours=duration_split_hours)
        )
        & (inputs_small_3["rate"].notnull())
    ]
    .copy()
    .reset_index(drop=True)
)

# Check if split is complete
assert len(df_temp1.index) + len(df_temp2.index) + len(df_temp3.index) + len(
    df_temp4.index
) == len(inputs_small_3.index)

In [ ]:
# We then process all of these dfs.
# In the first one, we need to duplicate the entries according to their duration and then divide each entry by the number of duplicates

# We duplicate the rows with the number bins for each injection
df_temp1["Repeat"] = np.ceil(
    (
        pd.to_datetime(df_temp1["endtime"]) - pd.to_datetime(df_temp1["starttime"])
    ).dt.total_seconds()
    / to_sec_fact
).astype(int)
df_new1 = df_temp1.reindex(df_temp1.index.repeat(df_temp1["Repeat"]))

In [ ]:
# We then create the admninistration time as a shifted version of the STARTTIME.
df_new1["charttime"] = df_new1.groupby(level=0)["starttime"].transform(
    lambda x: pd.date_range(
        start=x.iat[0], freq=str(60 * duration_split_hours) + "min", periods=len(x)
    )
)
# We divide each entry by the number of repeats
df_new1["amount"] = df_new1["amount"] / df_new1["Repeat"]

In [ ]:
# In the third one, we do the same
# We duplicate the rows with the number bins for each injection
df_temp3["Repeat"] = np.ceil(
    (
        pd.to_datetime(df_temp3["endtime"]) - pd.to_datetime(df_temp3["starttime"])
    ).dt.total_seconds()
    / to_sec_fact
).astype(int)
df_new3 = df_temp3.reindex(df_temp3.index.repeat(df_temp3["Repeat"]))
# We then create the admninistration time as a shifted version of the STARTTIME.

In [ ]:
df_new3["charttime"] = df_new3.groupby(level=0)["starttime"].transform(
    lambda x: pd.date_range(
        start=x.iat[0], freq=str(60 * duration_split_hours) + "min", periods=len(x)
    )
)
# We divide each entry by the number of repeats
df_new3["amount"] = df_new3["amount"] / df_new3["Repeat"]

df_temp2["charttime"] = df_temp2["starttime"]
df_temp4["charttime"] = df_temp4["starttime"]

In [ ]:
# Eventually, we merge all 4splits into one.
inputs_small_4 = pd.concat([df_new1, df_temp2, df_new3, df_temp4])
# The result is a dataset with discrete inputs for each treatment.

In [ ]:
inputs_small_4.to_csv("raw/inputs_processed.csv")
inputs_small_4["hadm_id"].nunique()

In [ ]:
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 300)

In [ ]:
adm = adm_3.copy()

In [ ]:
# Optimized labevents loading for large patient cohorts
# Only keep required columns and avoid repeated concat inside loop

adm_ids = set(adm["hadm_id"].dropna())

lab_chunks = []

for chunk in pd.read_csv(
    MIMIC_IV_ROOT / "hosp" / "labevents.csv.gz",
    chunksize=500000,
    usecols=["subject_id", "hadm_id", "charttime", "valuenum", "itemid"]
):
    # Remove events without admission id
    chunk = chunk.dropna(subset=["hadm_id"])

    # Filter target admissions
    chunk = chunk.loc[chunk["hadm_id"].isin(adm_ids)]

    if len(chunk) > 0:
        lab_chunks.append(chunk)

df = pd.concat(
    lab_chunks,
    ignore_index=True
)

print("Filtered labevents:", df.shape)


In [ ]:
# only choose previously selected admission ids.
print("Number of patients remaining in the database: ")
print(df["subject_id"].nunique())

In [ ]:
# get item ids
item_id = pd.read_csv("data/mimiciv/3.1/hosp/d_labitems.csv.gz")
item_id_1 = item_id[["itemid", "label"]]
item_id_1.head()

In [ ]:
# get names of administered items
lab2 = pd.merge(df, item_id_1, on="itemid")
lab2.head()
print("Number of patients remaining in the database: ")
print(lab2["subject_id"].nunique())

In [ ]:
lab2

In [ ]:
# get only top 150 most used tests
n_best = 150
pat_for_item = lab2.groupby("label")["subject_id"].nunique()
frequent_labels = pat_for_item.sort_values(ascending=False)[:n_best]
lab3 = lab2.loc[lab2["label"].isin(list(frequent_labels.index))].copy()

print("Number of patients remaining in the database: ")
print(lab3["subject_id"].nunique())

In [ ]:
lab3

In [ ]:
# only select the subset that was used in the paper (only missing is INR(PT))
subset = [
    "Albumin",
    "Alanine Aminotransferase (ALT)",
    "Alkaline Phosphatase",
    "Anion Gap",
    "Asparate Aminotransferase (AST)",
    "Base Excess",
    "Basophils",
    "Bicarbonate",
    "Bilirubin, Total",
    "Calcium, Total",
    "Calculated Total CO2",
    "Chloride",
    "Creatinine",
    "Eosinophils",
    "Glucose",
    "Hematocrit",
    "Hemoglobin",
    "Lactate",
    "Lymphocytes",
    "MCH",
    "MCV",
    "Magnesium",
    "Monocytes",
    "Neutrophils",
    "PT",
    "PTT",
    "Phosphate",
    "Platelet Count",
    "Potassium",
    "RDW",
    "Red Blood Cells",
    "Sodium",
    "Specific Gravity",
    "Urea Nitrogen",
    "White Blood Cells",
    "pCO2",
    "pH",
    "pO2",
]

lab4 = lab3.loc[lab3["label"].isin(subset)].copy()

In [ ]:
lab4

In [ ]:
lab4.to_csv("raw/lab_processed.csv")

In [ ]:
# Takes about 1 min
# only choose previously selected admission ids
presc = pd.read_csv(MIMIC_IV_ROOT / "hosp" / "prescriptions.csv.gz")
adm_ids = list(adm["hadm_id"])
presc = presc.loc[presc["hadm_id"].isin(adm_ids)]

print("Number of patients remaining in the database: ")
print(presc["subject_id"].nunique())
presc.tail()


In [ ]:
# Select entries whose drug name is in the list from the paper.
drugs_list = [
    "Acetaminophen",
    "Aspirin",
    "Bisacodyl",
    "Insulin",
    "Heparin",
    "Docusate Sodium",
    "D5W",
    "Humulin-R Insulin",
    "Potassium Chloride",
    "Magnesium Sulfate",
    "Metoprolol Tartrate",
    "Sodium Chloride 0.9%  Flush",
    "Pantoprazole",
]
presc2 = presc.loc[presc["drug"].isin(drugs_list)]

print("Number of patients remaining in the database: ")
print(presc2["subject_id"].nunique())

In [ ]:
print(presc2.groupby("drug")["dose_unit_rx"].value_counts())

In [ ]:
# Units correction
presc2 = presc2.drop(presc2.loc[presc2["dose_unit_rx"].isnull()].index).copy()
presc2 = presc2.drop(
    presc2.loc[
        (presc2["drug"] == "Acetaminophen") & (presc2["dose_unit_rx"] != "mg")
    ].index
).copy()
presc2.loc[
    (presc2["drug"] == "D5W") & (presc2["dose_unit_rx"] == "ml"), "dose_unit_rx"
] = "mL"
presc2 = presc2.drop(
    presc2.loc[(presc2["drug"] == "D5W") & (presc2["dose_unit_rx"] != "mL")].index
).copy()
presc2 = presc2.drop(
    presc2.loc[(presc2["drug"] == "Heparin") & (presc2["dose_unit_rx"] != "UNIT")].index
).copy()
presc2 = presc2.drop(
    presc2.loc[(presc2["drug"] == "Insulin") & (presc2["dose_unit_rx"] != "UNIT")].index
).copy()
presc2 = presc2.drop(
    presc2.loc[
        (presc2["drug"] == "Magnesium Sulfate") & (presc2["dose_unit_rx"] != "gm")
    ].index
).copy()
presc2 = presc2.drop(
    presc2.loc[
        (presc2["drug"] == "Potassium Chloride") & (presc2["dose_unit_rx"] != "mEq")
    ].index
).copy()
presc2.loc[
    (presc2["drug"] == "Sodium Chloride 0.9%  Flush")
    & (presc2["dose_unit_rx"] == "ml"),
    "dose_unit_rx",
] = "mL"
presc2 = presc2.drop(
    presc2.loc[(presc2["drug"] == "Bisacodyl") & (presc2["dose_unit_rx"] != "mg")].index
).copy()
presc2 = presc2.drop(
    presc2.loc[
        (presc2["drug"] == "Pantoprazole") & (presc2["dose_unit_rx"] != "mg")
    ].index
).copy()
print(presc2.groupby("drug")["dose_unit_rx"].value_counts())

In [ ]:
# To avoid confounding labels with labels from other tables, we add "drug" to the name
presc2["charttime"] = pd.to_datetime(presc2["starttime"], format="%Y-%m-%d %H:%M:%S")
presc2["drug"] = presc2["drug"] + " Drug"

In [ ]:
presc2.to_csv("raw/prescriptions_processed.csv")

In [ ]:
outputs = pd.read_csv(MIMIC_IV_ROOT / "icu" / "outputevents.csv.gz")
outputs.tail()


In [ ]:
# Restrict output events to the same patient-level cohort.
adm_ids = list(adm_3["hadm_id"])
outputs = outputs.loc[outputs["hadm_id"].isin(adm_ids)].copy()

print("Number of patients remaining in the database:")
print(outputs["subject_id"].nunique())
print("Number of datapoints remaining in the database:")
print(len(outputs.index))


In [ ]:
# get item names
item_id = pd.read_csv("data/mimiciv/3.1/icu/d_items.csv.gz")
item_id_1 = item_id[["itemid", "label"]]
item_id_1.head()

outputs_2 = pd.merge(outputs, item_id_1, on="itemid")
outputs_2.head()
print("Number of patients remaining in the database: ")
print(outputs_2["subject_id"].nunique())

In [ ]:
# take only the n most used items
n_best = 15
pat_for_item = outputs_2.groupby("label")["subject_id"].nunique()
frequent_labels = pat_for_item.sort_values(ascending=False)[:n_best]
outputs_3 = outputs_2.loc[outputs_2["label"].isin(list(frequent_labels.index))].copy()

print("Number of patients remaining in the database: ")
print(outputs_3["subject_id"].nunique())
print("Number of datapoints remaining in the database: ")
print(len(outputs_3.index))

print(frequent_labels)

In [ ]:
outputs_label_list = [
    "Foley",
    "Void",
    "OR Urine",
    "Chest Tube",
    "Oral Gastric",
    "Pre-Admission",
    "TF Residual",
    "OR EBL",
    "Emesis",
    "Nasogastric",
    "Stool",
    "Jackson Pratt",
    "TF Residual Output",
    "Fecal Bag",
    "Straight Cath",
]
outputs_bis = outputs_2.loc[outputs_2["label"].isin(outputs_label_list)].copy()

print("Number of patients remaining in the database: ")
print(outputs_bis["subject_id"].nunique())
print("Number of datapoints remaining in the database: ")
print(len(outputs_bis.index))

outputs_3 = outputs_bis.copy()

In [ ]:
# Verification that all input labels have the same amounts units
outputs_3.groupby("label")["valueuom"].value_counts()

In [ ]:
outputs_3.to_csv("raw/outputs_processed.csv")

In [ ]:
lab_df = pd.read_csv("raw/lab_processed.csv")[
    ["subject_id", "hadm_id", "charttime", "valuenum", "label"]
]
inputs_df = pd.read_csv("raw/inputs_processed.csv")[
    ["subject_id", "hadm_id", "charttime", "amount", "label"]
]
outputs_df = pd.read_csv("raw/outputs_processed.csv")[
    ["subject_id", "hadm_id", "charttime", "value", "label"]
]
presc_df = pd.read_csv("raw/prescriptions_processed.csv")[
    ["subject_id", "hadm_id", "charttime", "dose_val_rx", "drug"]
]

In [ ]:
# Change the name of amount. Valuenum for every table
inputs_df["valuenum"] = inputs_df["amount"]
inputs_df = inputs_df.drop(columns=["amount"]).copy()

In [ ]:
outputs_df["valuenum"] = outputs_df["value"]
outputs_df = outputs_df.drop(columns=["value"]).copy()

In [ ]:
presc_df["valuenum"] = presc_df["dose_val_rx"]
presc_df = presc_df.drop(columns=["dose_val_rx"]).copy()
presc_df["label"] = presc_df["drug"]
presc_df = presc_df.drop(columns=["drug"]).copy()
# Drop rows with non-numeric values
presc_df = presc_df.drop(
    presc_df[presc_df["valuenum"].str.contains("-", na=False)].index
)
presc_df["valuenum"] = (
    presc_df["valuenum"].astype(str).str.replace(",", "", regex=False)
)

In [ ]:
# Tag to distinguish between lab and inputs events
inputs_df["Origin"] = "Inputs"
lab_df["Origin"] = "Lab"
outputs_df["Origin"] = "Outputs"
presc_df["Origin"] = "Prescriptions"

In [ ]:
# merge both dfs.
merged_df1 = (pd.concat([inputs_df, lab_df])).reset_index(drop=True)
merged_df2 = (pd.concat([merged_df1, outputs_df])).reset_index(drop=True)
merged_df = (pd.concat([merged_df2, presc_df])).reset_index(drop=True)
assert merged_df["label"].nunique() == (
    inputs_df["label"].nunique()
    + lab_df["label"].nunique()
    + outputs_df["label"].nunique()
    + presc_df["label"].nunique()
)

In [ ]:
merged_df["charttime"] = pd.to_datetime(
    merged_df["charttime"], format="%Y-%m-%d %H:%M:%S"
)

In [ ]:
# =============================================================================
# CANONICAL TIME AXIS
# t=0 is ICU intime, not the first observed event.
# Keep exactly one 48h Core episode: [0, 24h) context + [24h, 48h) target.
# =============================================================================
merged_df["charttime"] = pd.to_datetime(merged_df["charttime"])

ref_time = adm_3.set_index("hadm_id")["intime"].copy()
merged_df_1 = pd.merge(
    ref_time.to_frame(name="ref_time"),
    merged_df,
    left_index=True,
    right_on="hadm_id",
    how="inner",
)

merged_df_1["time_stamp"] = merged_df_1["charttime"] - merged_df_1["ref_time"]
time_hours = merged_df_1["time_stamp"].dt.total_seconds() / 3600.0

merged_df_1 = merged_df_1[
    (time_hours >= 0)
    & (time_hours < TOTAL_WINDOW_HOURS)
].copy()

assert not (merged_df_1["time_stamp"] < timedelta(hours=0)).any()
print(
    f"Events retained in [0, {TOTAL_WINDOW_HOURS}h): "
    f"{len(merged_df_1):,}"
)


In [ ]:
# Create a label code (int) for the labels.
label_dict = dict(
    zip(
        list(merged_df_1["label"].unique()),
        range(len(list(merged_df_1["label"].unique()))),
    )
)
merged_df_1["label_code"] = merged_df_1["label"].map(label_dict)

merged_df_short = merged_df_1[
    ["hadm_id", "valuenum", "time_stamp", "label_code", "Origin"]
]

In [ ]:
label_dict_df = pd.Series(merged_df_1["label"].unique()).reset_index()
label_dict_df.columns = ["index", "label"]
label_dict_df["label_code"] = label_dict_df["label"].map(label_dict)
label_dict_df.drop(columns=["index"], inplace=True)
label_dict_df.to_csv("raw/label_dict.csv")

In [ ]:
merged_df_short["valuenum"] = merged_df_short["valuenum"].astype(float)

In [ ]:
# drop columns that are not needed for final dataset
merged_df_short.drop(["Origin"], axis=1, inplace=True)
merged_df_short.dropna(inplace=True)
complete_df = merged_df_short

In [ ]:

# Create Value and Mask columns using vectorized pivot (large cohort optimized)
# Avoid row-wise iterrows + .at assignment, which becomes extremely slow
# when scaling from TIME-IMM (20 patients) to TFS-IMM (thousands of patients).

label_codes = complete_df["label_code"].unique()

# Generate dense value matrix
value_df = complete_df.pivot_table(
    index=complete_df.index,
    columns="label_code",
    values="valuenum",
    aggfunc="first"
)

value_df.columns = [
    f"Value_label_{int(c)}" for c in value_df.columns
]

# Generate mask matrix
mask_df = value_df.notna().astype(int)
mask_df.columns = [
    c.replace("Value_", "Mask_") for c in mask_df.columns
]

# Merge back
complete_df = pd.concat(
    [
        complete_df,
        value_df,
        mask_df,
    ],
    axis=1
)

value_columns = list(value_df.columns)
mask_columns = list(mask_df.columns)

print("Created value columns:", len(value_columns))
print("Created mask columns:", len(mask_columns))


In [ ]:

# Row-wise filling removed.
# Previous implementation:
#   complete_df.iterrows()
#   complete_df.at[index, ...]
# is extremely slow for large MIMIC-IV cohorts.
#
# The pivot-based implementation above creates the same Value_label_xxx
# and Mask_label_xxx representation using vectorized pandas operations.


In [ ]:
# drop all unneccesary columns and do sanity check
complete_df.drop(["valuenum", "label_code"], axis=1, inplace=True)
# merge duplicate rows using hadim_id and time_stamp as keys
complete_df = complete_df.groupby(["hadm_id", "time_stamp"], as_index=False).max()
for x in mask_columns:
    assert len(complete_df.loc[complete_df[x] > 1]) == 0

In [ ]:
# Set Value_label_X to NaN where the corresponding mask is 0.
# Use dynamic columns instead of the original hard-coded range(109).
mask_cols = [col for col in complete_df.columns if col.startswith("Mask_label_")]

for mask_col in mask_cols:
    suffix = mask_col.replace("Mask_label_", "")
    value_col = f"Value_label_{suffix}"
    if value_col in complete_df.columns:
        complete_df.loc[complete_df[mask_col] == 0, value_col] = np.nan

# The benchmark keeps native missingness as NaN; masks are reconstructed by the
# downstream dataset loader from observed/non-observed values.
complete_df = complete_df.drop(columns=mask_cols)


In [ ]:
hadm_ids = complete_df["hadm_id"].values

Text Feature Preprocessing

In [ ]:

# =============================================================================
# TEXT MODALITY
# Keep radiology notes for compatibility with the public TIME-IMM preprocessing.
# Crucially, patients are NOT dropped merely because text is absent.
# =============================================================================

text_file = MIMIC_NOTE_ROOT / "radiology.csv.gz"
target_hadm_ids = set(adm_3["hadm_id"].dropna().astype(int).tolist())

rad_chunks = []

for chunk in pd.read_csv(
    text_file,
    chunksize=500_000,
    usecols=["hadm_id", "charttime", "text"]
):
    chunk = chunk.dropna(subset=["hadm_id"])
    chunk["hadm_id"] = chunk["hadm_id"].astype(int)

    chunk = chunk.loc[
        chunk["hadm_id"].isin(target_hadm_ids)
    ]

    if len(chunk) > 0:
        rad_chunks.append(chunk)

rad_df = pd.concat(
    rad_chunks,
    ignore_index=True
)

print(f"Radiology notes in selected admissions: {len(rad_df):,}")
print(f"Admissions with >=1 radiology note anywhere: {rad_df['hadm_id'].nunique():,}")


In [ ]:
# Align text to ICU intime and retain HISTORY text only.
# Any note at/after 24h is excluded from the model input to prevent future leakage.
rad_df["charttime"] = pd.to_datetime(rad_df["charttime"])

rad_df_1 = pd.merge(
    ref_time.to_frame(name="ref_time"),
    rad_df,
    left_index=True,
    right_on="hadm_id",
    how="inner",
)
rad_df_1["time_stamp"] = rad_df_1["charttime"] - rad_df_1["ref_time"]

text_hours = rad_df_1["time_stamp"].dt.total_seconds() / 3600.0
rad_df_1 = rad_df_1[
    (text_hours >= 0)
    & (text_hours < CONTEXT_HOURS)
].copy()

rad_df_1 = rad_df_1[["hadm_id", "time_stamp", "text"]]
rad_df_1 = rad_df_1.dropna(subset=["text"]).sort_values(["hadm_id", "time_stamp"])

print(f"History-window notes [0, {CONTEXT_HOURS}h): {len(rad_df_1):,}")
print(f"Patients with history text: {rad_df_1['hadm_id'].nunique():,}")


In [ ]:
# =============================================================================
# READ-ONLY TEXT-CATEGORY DIAGNOSTIC
# Estimate anatomical exam regions from the report EXAMINATION/PROCEDURE/STUDY
# header.  This cell does not alter rad_df_1 or any saved training data.
# Categories are same-level anatomical regions; for example, lung reports are
# counted under chest_lung rather than treated as a peer category of chest.
# =============================================================================
import re

_exam_header_re = re.compile(
    r"(?im)^\s*(?:examination|exam|procedure|study|type of exam|"
    r"radiographic examination)\s*:\s*(.+?)\s*$"
)
_region_patterns = {
    "chest_lung": re.compile(
        r"\b(chest|thorax|thoracic|lung|pulmonary|rib|sternum)\b", re.I
    ),
    "head_brain": re.compile(
        r"\b(head|brain|cranial|skull|facial|face|orbit|sinus)\b", re.I
    ),
    "neck": re.compile(r"\b(neck|thyroid)\b", re.I),
    "abdomen_pelvis": re.compile(
        r"\b(abdomen|abdominal|pelvis|pelvic|kub)\b", re.I
    ),
    "spine": re.compile(
        r"\b(spine|cervical|thoracolumbar|lumbar|lumbosacral|sacrum|coccyx)\b",
        re.I,
    ),
    "upper_extremity": re.compile(
        r"\b(shoulder|clavicle|humerus|elbow|forearm|wrist|hand|finger)\b",
        re.I,
    ),
    "lower_extremity": re.compile(
        r"\b(hip|femur|knee|tibia|fibula|ankle|foot|toe)\b", re.I
    ),
    "vascular": re.compile(
        r"\b(angiograph|angiogram|venogram|arteriogram|vascular)\b", re.I
    ),
    "breast": re.compile(r"\b(breast|mammogra)\w*\b", re.I),
    "whole_body": re.compile(r"\b(whole body|pet[- ]?ct)\b", re.I),
}


def _extract_exam_descriptor(text):
    text = "" if pd.isna(text) else str(text)
    match = _exam_header_re.search(text)
    if match:
        return match.group(1).strip()[:300]
    lines = [
        line.strip()
        for line in text.splitlines()[:20]
        if line.strip() and line.strip() != "___"
    ]
    return " ".join(lines[:3])[:300]


def _extract_regions(descriptor):
    matches = []
    for region, pattern in _region_patterns.items():
        match = pattern.search(descriptor)
        if match:
            matches.append((match.start(), region))
    return tuple(region for _, region in sorted(matches))


_text_category_diag = rad_df_1[["hadm_id", "text"]].copy()
_text_category_diag["exam_descriptor"] = (
    _text_category_diag["text"].map(_extract_exam_descriptor)
)
_text_category_diag["regions"] = (
    _text_category_diag["exam_descriptor"].map(_extract_regions)
)
_text_category_diag["primary_region"] = _text_category_diag["regions"].map(
    lambda values: values[0] if values else "unclassified"
)

_region_report_counts = (
    _text_category_diag.explode("regions")
    .dropna(subset=["regions"])
    .groupby("regions")
    .size()
    .sort_values(ascending=False)
)
_primary_region_summary = (
    _text_category_diag.groupby("primary_region")
    .agg(reports=("hadm_id", "size"), admissions=("hadm_id", "nunique"))
    .sort_values("reports", ascending=False)
)
_patient_region_count = (
    _text_category_diag.explode("regions")
    .dropna(subset=["regions"])
    .groupby("hadm_id")["regions"]
    .nunique()
)

print("Detected anatomical base categories:", len(_region_report_counts))
print("Reports with >1 detected region:",
      f"{(_text_category_diag['regions'].map(len) > 1).mean():.2%}")
print("Unclassified reports:",
      f"{(_text_category_diag['primary_region'] == 'unclassified').mean():.2%}")
print("Mean anatomical categories per classified admission:",
      f"{_patient_region_count.mean():.3f}")
print("Admissions with >=2 anatomical categories:",
      f"{(_patient_region_count >= 2).mean():.2%}")
display(_primary_region_summary)
display(_region_report_counts.rename("multi_label_report_count").to_frame())
display(
    _text_category_diag.loc[
        _text_category_diag["primary_region"] == "unclassified",
        ["exam_descriptor"],
    ].head(20)
)


Saving

In [ ]:
# Numeric-valid admissions define the Full cohort.
# Do NOT intersect with text IDs here: missing text is a valid modality state.
save_hadm_ids = np.unique(complete_df["hadm_id"].values).astype(int)
print(f"Admissions with numeric data in 48h episode: {len(save_hadm_ids):,}")


In [ ]:
folder_path = "raw/"
os.makedirs(folder_path, exist_ok=True)


In [ ]:
# Store all samples on a common synthetic datetime axis while preserving exact
# irregular relative timing. The downstream loader can convert this back to hours.
base_datetime = pd.Timestamp("2000-01-01 00:00:00")

complete_df["time_stamp"] = pd.to_timedelta(complete_df["time_stamp"]) + base_datetime
rad_df_1["time_stamp"] = pd.to_timedelta(rad_df_1["time_stamp"]) + base_datetime


In [ ]:
# Save text.csv for every numeric admission.
# Empty text.csv files are intentional for processed_full.
text_groups = {int(k): v.copy() for k, v in rad_df_1.groupby("hadm_id")}

for hadm_id_int in save_hadm_ids:
    folder_name = str(int(hadm_id_int))
    entity_dir = os.path.join(folder_path, folder_name)
    os.makedirs(entity_dir, exist_ok=True)
    file_path = os.path.join(entity_dir, "text.csv")

    if int(hadm_id_int) in text_groups:
        group_df = text_groups[int(hadm_id_int)].copy()
    else:
        group_df = pd.DataFrame(columns=["hadm_id", "time_stamp", "text"])

    group_df.to_csv(file_path, index=False)


In [ ]:
# Save the one-and-only 48h numeric episode for every admission.
ts_grouped_by_hadm_id = complete_df.groupby("hadm_id")

for hadm_id_float, group_df in ts_grouped_by_hadm_id:
    hadm_id_int = int(hadm_id_float)
    if hadm_id_int in set(save_hadm_ids):
        folder_name = str(hadm_id_int)
        entity_dir = os.path.join(folder_path, folder_name)
        os.makedirs(entity_dir, exist_ok=True)
        file_path = os.path.join(entity_dir, "time_series.csv")
        group_df.sort_values("time_stamp").to_csv(file_path, index=False)


In [ ]:
# =============================================================================
# RAW COHORT STATISTICS
# TIME-IMM selected only a small number of dense/long entities. TFS-IMM keeps
# the full eligible patient-level cohort and reports the distribution instead.
# =============================================================================
entity_stats = []

for hadm_id_int in save_hadm_ids:
    entity_id = str(int(hadm_id_int))
    entity_path = os.path.join("raw", entity_id)
    ts_path = os.path.join(entity_path, "time_series.csv")
    text_path = os.path.join(entity_path, "text.csv")

    if not os.path.exists(ts_path):
        continue

    df_ts = pd.read_csv(ts_path)
    feature_cols_all = [
        c for c in df_ts.columns
        if c.startswith("Value_label_")
    ]
    numeric_cells = df_ts[feature_cols_all].notna().sum().sum() if feature_cols_all else 0
    missing_rate = (
        df_ts[feature_cols_all].isna().sum().sum()
        / (len(df_ts) * len(feature_cols_all))
        if len(df_ts) and feature_cols_all else np.nan
    )

    text_rows = 0
    if os.path.exists(text_path):
        df_text = pd.read_csv(text_path)
        text_rows = len(df_text)

    entity_stats.append(
        {
            "entity_id": entity_id,
            "time_series_rows": len(df_ts),
            "numeric_observations": int(numeric_cells),
            "text_rows_0_24h": int(text_rows),
            "missing_rate_all_candidates": missing_rate,
        }
    )

entity_stats_df = pd.DataFrame(entity_stats)
entity_stats_df.to_csv("raw/entity_stats_all.csv", index=False)

print(f"Raw entities retained: {len(entity_stats_df):,}")
if len(entity_stats_df):
    print(entity_stats_df[[
        "time_series_rows",
        "numeric_observations",
        "text_rows_0_24h",
    ]].describe(percentiles=[0.25, 0.5, 0.75, 0.9]))


In [ ]:
# =============================================================================
# FIXED MULTIVARIATE FEATURE SPACE
# Retain the 30-feature TIME-IMM set for direct comparability.
# The feature space is global; individual patients are NOT required to observe
# every variable. Native missingness remains part of the benchmark.
# =============================================================================
important_feature_names = [
    "Dextrose 5%",
    "Sterile Water",
    "Fentanyl",
    "Heparin Sodium",
    "Solution",
    "Propofol",
    "Phenylephrine",
    "Foley",
    "Norepinephrine",
    "Midazolam (Versed)",
    "pH",
    "Base Excess",
    "Calculated Total CO2",
    "pCO2",
    "pO2",
    "Glucose",
    "Sodium",
    "Bicarbonate",
    "Chloride",
    "Urea Nitrogen",
    "Creatinine",
    "Potassium",
    "Vasopressin",
    "Anion Gap",
    "Magnesium",
    "Calcium, Total",
    "Phosphate",
    "Gastric Meds",
    "Insulin - Regular",
    "Piggyback",
]

missing_feature_names = [
    name for name in important_feature_names if name not in label_dict
]
if missing_feature_names:
    print("WARNING - configured features absent from current cohort:")
    print(missing_feature_names)

important_features = [
    "Value_label_" + str(label_dict[label])
    for label in important_feature_names
    if label in label_dict
]

feature_name_map = {
    "Value_label_" + str(label_dict[label]): label
    for label in important_feature_names
    if label in label_dict
}

print(f"Configured variables available: {len(important_features)}/{len(important_feature_names)}")


In [ ]:
# =============================================================================
# BUILD TFS-IMM CORE
# One patient -> one 48h episode.
# No overlapping sliding windows are generated.
# =============================================================================
import json
import shutil

FULL_ROOT = "processed_full"
# Keep strict multimodal samples under `processed/` for compatibility with
# the existing TIME-IMM ChunkedTimeSeriesDataset loader.
MM_ROOT = "processed"

if RESET_OUTPUT_DIRS:
    for root in [FULL_ROOT, MM_ROOT]:
        if os.path.isdir(root):
            shutil.rmtree(root)

for root in [FULL_ROOT, MM_ROOT]:
    os.makedirs(root, exist_ok=True)

hadm_to_subject = (
    adm_3[["hadm_id", "subject_id"]]
    .drop_duplicates("hadm_id")
    .set_index("hadm_id")["subject_id"]
    .to_dict()
)

def _relative_hours(date_series):
    return (
        pd.to_datetime(date_series) - base_datetime
    ).dt.total_seconds() / 3600.0

def _count_observed_variables(frame, cols):
    if frame.empty or not cols:
        return 0
    return int((frame[cols].notna().sum(axis=0) > 0).sum())

def _write_sample(root, entity_id, ts_df, text_df):
    entity_dir = os.path.join(root, str(entity_id))
    os.makedirs(entity_dir, exist_ok=True)
    ts_df.to_csv(os.path.join(entity_dir, "time_series.csv"), index=False)
    text_df.to_csv(os.path.join(entity_dir, "text.csv"), index=False)

sample_stats = []

for hadm_id_int in sorted(map(int, save_hadm_ids)):
    entity_id = str(hadm_id_int)
    raw_dir = os.path.join("raw", entity_id)
    ts_path = os.path.join(raw_dir, "time_series.csv")
    text_path = os.path.join(raw_dir, "text.csv")

    if not os.path.exists(ts_path):
        continue

    # --------------------------
    # Numeric 48h episode
    # --------------------------
    df_ts = pd.read_csv(ts_path)
    df_ts = df_ts.rename(
        columns={"time_stamp": "date_time", "hadm_id": "record_id"}
    )
    df_ts["date_time"] = pd.to_datetime(df_ts["date_time"])
    rel_h = _relative_hours(df_ts["date_time"])
    df_ts = df_ts[(rel_h >= 0) & (rel_h < TOTAL_WINDOW_HOURS)].copy()
    df_ts = df_ts.sort_values("date_time")

    # Fixed variable space. Missing variables remain all-NaN.
    for feature in important_features:
        if feature not in df_ts.columns:
            df_ts[feature] = np.nan

    cols_to_keep = ["date_time", "record_id"] + important_features
    df_ts = df_ts[cols_to_keep]

    rel_h = _relative_hours(df_ts["date_time"])
    hist = df_ts[(rel_h >= 0) & (rel_h < CONTEXT_HOURS)]
    target = df_ts[
        (rel_h >= CONTEXT_HOURS)
        & (rel_h < TOTAL_WINDOW_HOURS)
    ]

    hist_obs = int(hist[important_features].notna().sum().sum())
    target_obs = int(target[important_features].notna().sum().sum())
    hist_vars = _count_observed_variables(hist, important_features)
    target_vars = _count_observed_variables(target, important_features)

    numeric_valid = (
        hist_obs >= MIN_HISTORY_OBSERVATIONS
        and target_obs >= MIN_TARGET_OBSERVATIONS
        and hist_vars >= MIN_HISTORY_VARIABLES
        and target_vars >= MIN_TARGET_VARIABLES
    )

    # --------------------------
    # History text only
    # --------------------------
    if os.path.exists(text_path):
        df_text = pd.read_csv(text_path)
    else:
        df_text = pd.DataFrame(columns=["hadm_id", "time_stamp", "text"])

    df_text = df_text.rename(
        columns={"time_stamp": "date_time", "hadm_id": "record_id"}
    )

    if "date_time" not in df_text.columns:
        df_text["date_time"] = pd.Series(dtype="datetime64[ns]")
    if "record_id" not in df_text.columns:
        df_text["record_id"] = hadm_id_int
    if "text" not in df_text.columns:
        df_text["text"] = pd.Series(dtype=str)

    if len(df_text):
        df_text["date_time"] = pd.to_datetime(df_text["date_time"])
        text_rel_h = _relative_hours(df_text["date_time"])
        df_text = df_text[
            (text_rel_h >= 0)
            & (text_rel_h < CONTEXT_HOURS)
        ].copy()
        df_text = df_text.dropna(subset=["text"]).sort_values("date_time")

    df_text = df_text[["date_time", "record_id", "text"]]
    text_count = int(len(df_text))
    text_chars = int(df_text["text"].astype(str).str.len().sum()) if text_count else 0

    subject_id = int(hadm_to_subject[hadm_id_int])

    sample_stats.append(
        {
            "subject_id": subject_id,
            "record_id": hadm_id_int,
            "numeric_valid": bool(numeric_valid),
            "history_observations": hist_obs,
            "target_observations": target_obs,
            "history_variables": hist_vars,
            "target_variables": target_vars,
            "text_count_0_24h": text_count,
            "text_chars_0_24h": text_chars,
            "has_text": bool(text_count >= MIN_TEXT_NOTES_MM),
        }
    )

    if not numeric_valid:
        continue

    # Full cohort: keep text-present and text-absent samples.
    _write_sample(FULL_ROOT, entity_id, df_ts, df_text)

    # Strict multimodal subset.
    if text_count >= MIN_TEXT_NOTES_MM:
        _write_sample(MM_ROOT, entity_id, df_ts, df_text)

sample_stats_df = pd.DataFrame(sample_stats)
sample_stats_df.to_csv("sample_statistics.csv", index=False)

valid_stats = sample_stats_df[sample_stats_df["numeric_valid"]].copy()
full_ids = valid_stats["record_id"].astype(int).tolist()
mm_ids = valid_stats.loc[valid_stats["has_text"], "record_id"].astype(int).tolist()

print(f"Numeric-valid Full cohort: {len(full_ids):,}")
print(f"Strict MM cohort (>= {MIN_TEXT_NOTES_MM} note): {len(mm_ids):,}")

if len(valid_stats):
    text_coverage = valid_stats["has_text"].mean()
    print(f"History-text coverage: {text_coverage:.2%}")
    print("Notes per valid sample:")
    print(
        valid_stats["text_count_0_24h"].describe(
            percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
        )
    )

# Dataset configuration for reproducibility.
# NOTE: Train/val/test splitting and numerical normalization are intentionally
# NOT performed in this preprocessing notebook. Keep them in the downstream
# dataset loader/training pipeline, matching the original TIME-IMM workflow.
dataset_config = {
    "context_hours": CONTEXT_HOURS,
    "prediction_hours": PRED_HOURS,
    "one_episode_per_patient": True,
    "sliding_window": False,
    "min_age": MIN_AGE,
    "min_icu_stay_hours": MIN_ICU_STAY_HOURS,
    "max_icu_stay_days": MAX_ICU_STAY_DAYS,
    "num_configured_features": len(important_features),
    "feature_names": [feature_name_map.get(f, f) for f in important_features],
    "text_source": TEXT_SOURCE,
    "text_history_only": True,
    "min_text_notes_mm": MIN_TEXT_NOTES_MM,
    "processed_full_root": FULL_ROOT,
    "processed_multimodal_root": MM_ROOT,
}
with open("tfsimm_dataset_config.json", "w", encoding="utf-8") as f:
    json.dump(dataset_config, f, indent=2, ensure_ascii=False)

print("TFS-IMM preprocessing definition written successfully.")
